# Modeling Exam (Easy) — CoderPad Practice Problems

Confidence builders for ML modeling fundamentals. 15-20 min each. If these take longer, drill them before moving to MEDIUM.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import math
from typing import Optional, Tuple

---

## Problem 1: Linear Layer from Scratch

**Scenario:** The linear (fully-connected) layer is the most fundamental building block in deep learning. You should be able to implement it from memory.

**Spec:**

Implement `ScratchLinear(nn.Module)` with:

**`__init__(self, in_features, out_features, bias=True)`**
- `weight` — `nn.Parameter` of shape `(out_features, in_features)`, initialized from `U(-1/sqrt(in_features), 1/sqrt(in_features))`
- `bias` — `nn.Parameter` of shape `(out_features,)` with the same initialization, or `None` if `bias=False`

**`forward(self, x)`**
- Compute `x @ W^T + b` (or just `x @ W^T` if no bias)
- Input shape: `(*, in_features)` → Output shape: `(*, out_features)`

**Constraints:**
- Do NOT use `nn.Linear`.
- All parameters must be properly registered (visible via `model.parameters()`).

In [ ]:
class ScratchLinear(nn.Module):
    """Linear layer implemented from scratch.

    Args:
        in_features: Size of each input sample.
        out_features: Size of each output sample.
        bias: If True, adds a learnable bias.
    """
    # YOUR CODE HERE
    pass

In [ ]:
# --- Tests: Problem 1 ---
torch.manual_seed(42)

# Basic forward pass
layer = ScratchLinear(10, 5, bias=True)
x = torch.randn(3, 10)
out = layer(x)
assert out.shape == (3, 5), f"Expected (3, 5), got {out.shape}"

# Weight and bias shapes
assert layer.weight.shape == (5, 10), f"Weight shape should be (5, 10), got {layer.weight.shape}"
assert layer.bias.shape == (5,), f"Bias shape should be (5,), got {layer.bias.shape}"

# Parameters are registered
param_count = sum(p.numel() for p in layer.parameters())
assert param_count == 55, f"Expected 55 params (50 weight + 5 bias), got {param_count}"

# No bias variant
layer_nb = ScratchLinear(8, 4, bias=False)
assert layer_nb.bias is None, "bias should be None when bias=False"
nb_params = sum(p.numel() for p in layer_nb.parameters())
assert nb_params == 32, f"Expected 32 params (no bias), got {nb_params}"
out_nb = layer_nb(torch.randn(2, 8))
assert out_nb.shape == (2, 4)

# Verify it computes x @ W^T + b
torch.manual_seed(0)
layer_check = ScratchLinear(4, 3, bias=True)
x_check = torch.randn(2, 4)
expected = x_check @ layer_check.weight.T + layer_check.bias
actual = layer_check(x_check)
assert torch.allclose(actual, expected, atol=1e-6), "Forward should compute x @ W^T + b"

# Batched input (3D)
x_3d = torch.randn(2, 3, 10)
out_3d = layer(x_3d)
assert out_3d.shape == (2, 3, 5), f"Should handle batched input, got {out_3d.shape}"

# Gradient flows
loss = out.sum()
loss.backward()
assert layer.weight.grad is not None
assert layer.bias.grad is not None

print("Problem 1: ALL TESTS PASSED")

---

## Problem 2: Activation Functions

**Scenario:** Activation functions introduce non-linearity. You should know the formulas for the most common ones and be able to implement them from scratch.

**Spec:**

Implement three activation functions as standalone functions (not modules):

**`scratch_relu(x)`** — `max(0, x)` element-wise

**`scratch_gelu(x)`** — Gaussian Error Linear Unit (tanh approximation):

$$\text{GELU}(x) = x \cdot 0.5 \cdot \left(1 + \tanh\left(\sqrt{\frac{2}{\pi}} \cdot (x + 0.044715 \cdot x^3)\right)\right)$$

**`scratch_swish(x)`** — Swish / SiLU:

$$\text{Swish}(x) = x \cdot \sigma(x) = \frac{x}{1 + e^{-x}}$$

**Constraints:**
- Do NOT use `F.relu`, `F.gelu`, `F.silu`, or `torch.relu` etc.
- Use only basic tensor ops (`torch.tanh`, `torch.sigmoid`, `torch.clamp`, arithmetic).

In [ ]:
def scratch_relu(x: torch.Tensor) -> torch.Tensor:
    """ReLU activation from scratch."""
    # YOUR CODE HERE
    pass


def scratch_gelu(x: torch.Tensor) -> torch.Tensor:
    """GELU activation from scratch (tanh approximation)."""
    # YOUR CODE HERE
    pass


def scratch_swish(x: torch.Tensor) -> torch.Tensor:
    """Swish (SiLU) activation from scratch."""
    # YOUR CODE HERE
    pass

In [ ]:
# --- Tests: Problem 2 ---
torch.manual_seed(42)
x = torch.randn(100)

# ReLU: negative values become 0, positive unchanged
r = scratch_relu(x)
assert (r >= 0).all(), "ReLU output should be non-negative"
assert torch.allclose(r, torch.clamp(x, min=0), atol=1e-6)
assert scratch_relu(torch.tensor([-1.0])).item() == 0.0
assert scratch_relu(torch.tensor([3.0])).item() == 3.0

# GELU: compare against PyTorch reference (tanh approximation)
g = scratch_gelu(x)
g_ref = F.gelu(x, approximate="tanh")
assert torch.allclose(g, g_ref, atol=1e-5), f"GELU max diff: {(g - g_ref).abs().max()}"
assert scratch_gelu(torch.tensor([0.0])).item() == 0.0  # GELU(0) = 0

# Swish: compare against PyTorch SiLU
s = scratch_swish(x)
s_ref = F.silu(x)
assert torch.allclose(s, s_ref, atol=1e-5), f"Swish max diff: {(s - s_ref).abs().max()}"
assert scratch_swish(torch.tensor([0.0])).item() == 0.0  # Swish(0) = 0

# Shape preservation
x_2d = torch.randn(4, 8)
assert scratch_relu(x_2d).shape == (4, 8)
assert scratch_gelu(x_2d).shape == (4, 8)
assert scratch_swish(x_2d).shape == (4, 8)

# Swish is NOT the same as ReLU (swish can be slightly negative)
assert (scratch_swish(torch.tensor([-1.0])) < 0).item(), "Swish(-1) should be slightly negative"

print("Problem 2: ALL TESTS PASSED")

---

## Problem 3: Sinusoidal Positional Encoding

**Scenario:** Diffusion models condition on a scalar timestep `t`. The standard approach is to encode `t` into a high-dimensional vector using sinusoidal positional encoding (same idea as the original Transformer).

**Spec:**

Implement `sinusoidal_encoding(timesteps, dim)` that maps scalar timesteps to dense vectors.

**Inputs:**
- `timesteps` — tensor of shape `(B,)` with integer or float timestep values
- `dim` — int, embedding dimension (must be even)

**Output:** tensor of shape `(B, dim)`

**Formulas:**

$$\text{PE}(\text{pos}, 2i) = \sin\left(\frac{\text{pos}}{10000^{2i/\text{dim}}}\right)$$

$$\text{PE}(\text{pos}, 2i+1) = \cos\left(\frac{\text{pos}}{10000^{2i/\text{dim}}}\right)$$

where `i` ranges from `0` to `dim/2 - 1`.

**Constraints:**
- No loops over the batch or dimension. Use broadcasting.
- Return float32 tensor.

In [ ]:
def sinusoidal_encoding(timesteps: torch.Tensor, dim: int) -> torch.Tensor:
    """Compute sinusoidal positional encoding for timesteps.

    Args:
        timesteps: (B,) tensor of timestep values.
        dim: Embedding dimension (must be even).

    Returns:
        (B, dim) tensor of positional encodings.
    """
    # YOUR CODE HERE
    pass

In [ ]:
# --- Tests: Problem 3 ---

# Basic shape
t = torch.tensor([0, 1, 2, 100], dtype=torch.float32)
enc = sinusoidal_encoding(t, dim=64)
assert enc.shape == (4, 64), f"Expected (4, 64), got {enc.shape}"
assert enc.dtype == torch.float32

# Timestep 0: sin(0)=0 for even indices, cos(0)=1 for odd indices
enc_zero = sinusoidal_encoding(torch.tensor([0.0]), dim=8)
assert torch.allclose(enc_zero[0, 0::2], torch.zeros(4), atol=1e-6), "sin(0) should be 0"
assert torch.allclose(enc_zero[0, 1::2], torch.ones(4), atol=1e-6), "cos(0) should be 1"

# Values are bounded in [-1, 1]
enc_large = sinusoidal_encoding(torch.arange(1000, dtype=torch.float32), dim=128)
assert enc_large.min() >= -1.0 - 1e-6
assert enc_large.max() <= 1.0 + 1e-6

# Different timesteps produce different encodings
enc_diff = sinusoidal_encoding(torch.tensor([0.0, 1.0]), dim=32)
assert not torch.allclose(enc_diff[0], enc_diff[1]), "Different timesteps should have different encodings"

# Same timestep produces same encoding (deterministic)
enc_a = sinusoidal_encoding(torch.tensor([42.0]), dim=64)
enc_b = sinusoidal_encoding(torch.tensor([42.0]), dim=64)
assert torch.allclose(enc_a, enc_b)

# Low-frequency dimensions change slowly, high-frequency dimensions change fast
enc_seq = sinusoidal_encoding(torch.arange(10, dtype=torch.float32), dim=64)
# First sin dimension (i=0) has highest frequency
# Last sin dimension (i=31) has lowest frequency
diff_first = (enc_seq[1:, 0] - enc_seq[:-1, 0]).abs().mean()
diff_last = (enc_seq[1:, -2] - enc_seq[:-1, -2]).abs().mean()
assert diff_first > diff_last, "Low-index dims should change faster than high-index dims"

# Single timestep
enc_single = sinusoidal_encoding(torch.tensor([5.0]), dim=16)
assert enc_single.shape == (1, 16)

print("Problem 3: ALL TESTS PASSED")

---

## Problem 4: Batch Normalization Forward Pass

**Scenario:** Batch normalization is standard in CNNs and some transformer variants. Understanding the forward pass mechanics (especially the difference between train and eval modes) is essential.

**Spec:**

Implement `batch_norm_forward(x, gamma, beta, running_mean, running_var, training, momentum=0.1, eps=1e-5)` for 4D input `(B, C, H, W)`.

**Training mode:**
1. Compute batch mean and variance per channel (over B, H, W dims)
2. Normalize: $\hat{x} = \frac{x - \mu_{\text{batch}}}{\sqrt{\sigma^2_{\text{batch}} + \epsilon}}$
3. Scale and shift: $y = \gamma \cdot \hat{x} + \beta$
4. Update running stats:
   - `running_mean = (1 - momentum) * running_mean + momentum * batch_mean`
   - `running_var = (1 - momentum) * running_var + momentum * batch_var`

**Eval mode:**
- Use `running_mean` and `running_var` instead of batch statistics.
- Do NOT update running stats.

**Returns:** `(output, running_mean, running_var)` — all tensors.

**Constraints:**
- Do NOT use `nn.BatchNorm2d` or `F.batch_norm`.
- `gamma` and `beta` are shape `(C,)`. Reshape for broadcasting.

In [ ]:
def batch_norm_forward(
    x: torch.Tensor,
    gamma: torch.Tensor,
    beta: torch.Tensor,
    running_mean: torch.Tensor,
    running_var: torch.Tensor,
    training: bool,
    momentum: float = 0.1,
    eps: float = 1e-5,
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Batch normalization forward pass.

    Args:
        x: Input tensor of shape (B, C, H, W).
        gamma: Scale parameter of shape (C,).
        beta: Shift parameter of shape (C,).
        running_mean: Running mean of shape (C,).
        running_var: Running variance of shape (C,).
        training: Whether in training mode.
        momentum: Momentum for running stats update.
        eps: Epsilon for numerical stability.

    Returns:
        (output, updated_running_mean, updated_running_var)
    """
    # YOUR CODE HERE
    pass

In [ ]:
# --- Tests: Problem 4 ---
torch.manual_seed(42)

B, C, H, W = 8, 3, 4, 4
x = torch.randn(B, C, H, W)
gamma = torch.ones(C)
beta = torch.zeros(C)
rm = torch.zeros(C)
rv = torch.ones(C)

# Training mode: output should be normalized per channel
out, rm_new, rv_new = batch_norm_forward(x, gamma, beta, rm.clone(), rv.clone(), training=True)
assert out.shape == (B, C, H, W), f"Output shape should be (B,C,H,W), got {out.shape}"

# With gamma=1, beta=0: per-channel mean should be ~0, std ~1
for c in range(C):
    channel_out = out[:, c, :, :]
    assert abs(channel_out.mean().item()) < 1e-5, f"Channel {c} mean should be ~0"
    assert abs(channel_out.std().item() - 1.0) < 0.1, f"Channel {c} std should be ~1"

# Running stats should be updated
assert not torch.allclose(rm_new, torch.zeros(C)), "Running mean should be updated in training"

# Running mean update: (1-0.1)*0 + 0.1*batch_mean
batch_mean = x.mean(dim=(0, 2, 3))
expected_rm = 0.9 * torch.zeros(C) + 0.1 * batch_mean
assert torch.allclose(rm_new, expected_rm, atol=1e-5)

# Eval mode: uses running stats, does NOT update them
rm_eval = rm_new.clone()
rv_eval = rv_new.clone()
out_eval, rm_eval_after, rv_eval_after = batch_norm_forward(
    x, gamma, beta, rm_eval, rv_eval, training=False
)
assert out_eval.shape == (B, C, H, W)
assert torch.allclose(rm_eval_after, rm_eval), "Running stats should NOT change in eval mode"
assert torch.allclose(rv_eval_after, rv_eval), "Running stats should NOT change in eval mode"

# Non-trivial gamma/beta
gamma2 = torch.tensor([2.0, 0.5, 3.0])
beta2 = torch.tensor([1.0, -1.0, 0.5])
out2, _, _ = batch_norm_forward(x, gamma2, beta2, rm.clone(), rv.clone(), training=True)
for c in range(C):
    channel_out = out2[:, c, :, :]
    assert abs(channel_out.mean().item() - beta2[c].item()) < 0.1, "Mean should be ~beta"

print("Problem 4: ALL TESTS PASSED")

---

## Problem 5: Basic Conv2d

**Scenario:** Convolutional layers are the backbone of image and video models. You should understand how to set up the weight parameter and how spatial dimensions change.

**Spec:**

**Part A:** Implement `ScratchConv2d(nn.Module)` — a 2D convolution layer.

**`__init__(self, in_channels, out_channels, kernel_size)`**
- `weight` — `nn.Parameter` of shape `(out_channels, in_channels, kernel_size, kernel_size)`
- Initialize with `nn.init.kaiming_uniform_(weight, a=math.sqrt(5))`
- No bias, stride=1, padding=0

**`forward(self, x)`**
- Use `F.conv2d` with your weight parameter. (The point is managing the parameter, not reimplementing convolution.)

**Part B:** Implement `conv_output_size(h, w, kernel_size, stride=1, padding=0)` that returns `(h_out, w_out)`.

**Formula:**

$$H_{\text{out}} = \left\lfloor \frac{H + 2 \times \text{padding} - \text{kernel\_size}}{\text{stride}} + 1 \right\rfloor$$

(Same formula for width.)

In [ ]:
class ScratchConv2d(nn.Module):
    """Conv2d layer with manually managed weight parameter.

    Args:
        in_channels: Number of input channels.
        out_channels: Number of output channels.
        kernel_size: Size of the convolving kernel.
    """
    # YOUR CODE HERE
    pass


def conv_output_size(
    h: int, w: int, kernel_size: int, stride: int = 1, padding: int = 0
) -> Tuple[int, int]:
    """Compute output spatial dimensions for a convolution.

    Args:
        h: Input height.
        w: Input width.
        kernel_size: Kernel size.
        stride: Stride.
        padding: Padding.

    Returns:
        (h_out, w_out)
    """
    # YOUR CODE HERE
    pass

In [ ]:
# --- Tests: Problem 5 ---
torch.manual_seed(42)

# Part A: ScratchConv2d
conv = ScratchConv2d(3, 16, kernel_size=3)
x = torch.randn(2, 3, 8, 8)
out = conv(x)
assert out.shape == (2, 16, 6, 6), f"Expected (2, 16, 6, 6), got {out.shape}"

# Weight shape
assert conv.weight.shape == (16, 3, 3, 3), f"Weight shape wrong: {conv.weight.shape}"

# Parameter registration
param_count = sum(p.numel() for p in conv.parameters())
assert param_count == 16 * 3 * 3 * 3, f"Expected {16*3*3*3} params, got {param_count}"

# 1x1 convolution
conv1x1 = ScratchConv2d(8, 4, kernel_size=1)
out1x1 = conv1x1(torch.randn(1, 8, 16, 16))
assert out1x1.shape == (1, 4, 16, 16), "1x1 conv should preserve spatial dims"

# Gradient flows
loss = out.sum()
loss.backward()
assert conv.weight.grad is not None

# Part B: conv_output_size
assert conv_output_size(8, 8, kernel_size=3) == (6, 6)
assert conv_output_size(32, 32, kernel_size=3, stride=1, padding=1) == (32, 32)  # same padding
assert conv_output_size(32, 32, kernel_size=3, stride=2, padding=1) == (16, 16)  # stride 2
assert conv_output_size(28, 28, kernel_size=5, stride=1, padding=0) == (24, 24)
assert conv_output_size(7, 7, kernel_size=7, stride=1, padding=0) == (1, 1)      # global conv
assert conv_output_size(224, 224, kernel_size=7, stride=2, padding=3) == (112, 112)  # ResNet stem

# Non-square
assert conv_output_size(10, 20, kernel_size=3, stride=2, padding=0) == (4, 9)

print("Problem 5: ALL TESTS PASSED")

---

## Problem 6: MSE, MAE, and Huber Loss

**Scenario:** Loss functions are the objective you optimize. In diffusion models, MSE (L2) is the standard noise prediction loss. MAE (L1) and Huber are common alternatives with different sensitivity to outliers.

**Spec:**

Implement three loss functions. Each takes `predictions` and `targets` of the same shape and returns a scalar mean loss.

**`scratch_mse(pred, target)`** — Mean Squared Error:
$$L = \frac{1}{N} \sum (\text{pred} - \text{target})^2$$

**`scratch_mae(pred, target)`** — Mean Absolute Error:
$$L = \frac{1}{N} \sum |\text{pred} - \text{target}|$$

**`scratch_huber(pred, target, delta=1.0)`** — Huber Loss (smooth L1):
$$L(x) = \begin{cases} 0.5 \cdot x^2 & \text{if } |x| \leq \delta \\ \delta \cdot (|x| - 0.5 \cdot \delta) & \text{otherwise} \end{cases}$$
where $x = \text{pred} - \text{target}$. Return the mean.

**Constraints:**
- Do NOT use `F.mse_loss`, `F.l1_loss`, `F.huber_loss`, or `F.smooth_l1_loss`.
- Each returns a scalar (0-dim tensor).

In [ ]:
def scratch_mse(pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    """Mean Squared Error loss."""
    # YOUR CODE HERE
    pass


def scratch_mae(pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    """Mean Absolute Error loss."""
    # YOUR CODE HERE
    pass


def scratch_huber(
    pred: torch.Tensor, target: torch.Tensor, delta: float = 1.0
) -> torch.Tensor:
    """Huber (smooth L1) loss."""
    # YOUR CODE HERE
    pass

In [ ]:
# --- Tests: Problem 6 ---
torch.manual_seed(42)

pred = torch.randn(16, 8)
target = torch.randn(16, 8)

# MSE matches reference
mse = scratch_mse(pred, target)
assert mse.dim() == 0, "MSE should return scalar"
assert torch.allclose(mse, F.mse_loss(pred, target), atol=1e-5)

# MSE is zero for identical inputs
assert scratch_mse(pred, pred).item() == 0.0

# MAE matches reference
mae = scratch_mae(pred, target)
assert mae.dim() == 0, "MAE should return scalar"
assert torch.allclose(mae, F.l1_loss(pred, target), atol=1e-5)

# MAE is zero for identical inputs
assert scratch_mae(pred, pred).item() == 0.0

# Huber matches reference (delta=1.0)
huber = scratch_huber(pred, target, delta=1.0)
assert huber.dim() == 0, "Huber should return scalar"
assert torch.allclose(huber, F.huber_loss(pred, target, delta=1.0), atol=1e-5)

# Huber with different delta
huber2 = scratch_huber(pred, target, delta=0.5)
assert torch.allclose(huber2, F.huber_loss(pred, target, delta=0.5), atol=1e-5)

# Huber acts like MSE/2 for small errors
small_pred = torch.tensor([0.1])
small_target = torch.tensor([0.0])
assert torch.allclose(
    scratch_huber(small_pred, small_target, delta=1.0),
    torch.tensor(0.005)  # 0.5 * 0.1^2 = 0.005
)

# All losses are non-negative
assert mse >= 0
assert mae >= 0
assert huber >= 0

# MSE penalizes outliers more than MAE
outlier_pred = torch.tensor([10.0])
outlier_target = torch.tensor([0.0])
assert scratch_mse(outlier_pred, outlier_target) > scratch_mae(outlier_pred, outlier_target)

print("Problem 6: ALL TESTS PASSED")

---

## Problem 7: Simple Residual Block

**Scenario:** Residual (skip) connections are what make deep networks trainable. ResNets, transformers, U-Nets, and diffusion models all rely on them. You should be able to write a residual block from memory.

**Spec:**

Implement `ResidualBlock(nn.Module)` with:

**`__init__(self, dim)`**
- Two `nn.Linear` layers: `linear1(dim, dim)` and `linear2(dim, dim)`

**`forward(self, x)`**
$$\text{output} = x + \text{ReLU}(\text{linear2}(\text{ReLU}(\text{linear1}(x))))$$

The skip connection adds the original input to the output of the two-layer sub-network.

**Constraints:**
- Input and output have the same shape `(B, dim)`.
- Use `F.relu` for activation (it's fine here — the point is the residual pattern).

In [ ]:
class ResidualBlock(nn.Module):
    """Simple residual block with two linear layers.

    Args:
        dim: Input and output dimension.
    """
    # YOUR CODE HERE
    pass

In [ ]:
# --- Tests: Problem 7 ---
torch.manual_seed(42)

# Basic forward pass
block = ResidualBlock(dim=32)
x = torch.randn(4, 32)
out = block(x)
assert out.shape == (4, 32), f"Expected (4, 32), got {out.shape}"

# Output is NOT the same as input (unless weights are pathological)
assert not torch.allclose(out, x, atol=1e-3), "Output should differ from input"

# Verify skip connection: if we zero out the linear layers, output should equal input
block_zero = ResidualBlock(dim=16)
with torch.no_grad():
    block_zero.linear1.weight.zero_()
    block_zero.linear1.bias.zero_()
    block_zero.linear2.weight.zero_()
    block_zero.linear2.bias.zero_()
x_test = torch.randn(3, 16)
out_zero = block_zero(x_test)
assert torch.allclose(out_zero, x_test), "With zeroed weights, output should equal input (skip only)"

# Parameter count: two linear layers of (dim, dim) with biases
param_count = sum(p.numel() for p in block.parameters())
# linear1: 32*32 + 32 = 1056, linear2: 32*32 + 32 = 1056, total = 2112
assert param_count == 2112, f"Expected 2112 params, got {param_count}"

# Gradient flows through both the skip and the residual path
x_grad = torch.randn(2, 32, requires_grad=True)
out_grad = block(x_grad)
out_grad.sum().backward()
assert x_grad.grad is not None, "Gradients should flow to input"
# Gradient should not be exactly 1 (would mean residual path contributes nothing)
assert not torch.allclose(x_grad.grad, torch.ones_like(x_grad.grad)), \
    "Gradient should have contributions from residual path"

# Has two linear layers
assert hasattr(block, 'linear1') and hasattr(block, 'linear2')
assert isinstance(block.linear1, nn.Linear)
assert isinstance(block.linear2, nn.Linear)

print("Problem 7: ALL TESTS PASSED")

---

## Problem 8: Learning Rate Scheduler

**Scenario:** Most modern training runs use warmup + cosine decay. Understanding the math lets you debug training curves and implement custom schedules.

**Spec:**

Implement `warmup_cosine_lr(current_step, warmup_steps, total_steps, base_lr)` that returns the learning rate for a given step.

**Warmup phase** (`current_step < warmup_steps`):
$$\text{lr} = \text{base\_lr} \times \frac{\text{current\_step}}{\text{warmup\_steps}}$$

Linear ramp from 0 to `base_lr`.

**Cosine decay phase** (`current_step >= warmup_steps`):
$$\text{lr} = \text{base\_lr} \times 0.5 \times \left(1 + \cos\left(\pi \times \frac{\text{current\_step} - \text{warmup\_steps}}{\text{total\_steps} - \text{warmup\_steps}}\right)\right)$$

Smooth decay from `base_lr` to 0.

**Constraints:**
- Returns a Python float.
- `current_step` can be 0 to `total_steps - 1`.
- At `current_step == 0`: lr = 0 (start of warmup).
- At `current_step == warmup_steps`: lr = base_lr (warmup just finished).

In [ ]:
def warmup_cosine_lr(
    current_step: int,
    warmup_steps: int,
    total_steps: int,
    base_lr: float,
) -> float:
    """Compute learning rate with linear warmup + cosine decay.

    Args:
        current_step: Current training step (0-indexed).
        warmup_steps: Number of warmup steps.
        total_steps: Total number of training steps.
        base_lr: Peak learning rate.

    Returns:
        Learning rate for the current step.
    """
    # YOUR CODE HERE
    pass

In [ ]:
# --- Tests: Problem 8 ---

base_lr = 1e-3
warmup_steps = 100
total_steps = 1000

# Step 0: lr = 0 (start of warmup)
assert warmup_cosine_lr(0, warmup_steps, total_steps, base_lr) == 0.0

# Halfway through warmup: lr = base_lr * 0.5
lr_mid_warmup = warmup_cosine_lr(50, warmup_steps, total_steps, base_lr)
assert abs(lr_mid_warmup - base_lr * 0.5) < 1e-10

# At warmup_steps: lr = base_lr (peak)
lr_peak = warmup_cosine_lr(warmup_steps, warmup_steps, total_steps, base_lr)
assert abs(lr_peak - base_lr) < 1e-10, f"At warmup end, lr should be base_lr, got {lr_peak}"

# At total_steps - 1: lr should be very close to 0
lr_end = warmup_cosine_lr(total_steps - 1, warmup_steps, total_steps, base_lr)
assert lr_end < base_lr * 0.01, f"At end, lr should be near 0, got {lr_end}"

# LR is monotonically increasing during warmup
warmup_lrs = [warmup_cosine_lr(s, warmup_steps, total_steps, base_lr) for s in range(warmup_steps)]
for i in range(1, len(warmup_lrs)):
    assert warmup_lrs[i] > warmup_lrs[i - 1], "LR should increase during warmup"

# LR is monotonically decreasing during cosine decay
decay_lrs = [warmup_cosine_lr(s, warmup_steps, total_steps, base_lr) for s in range(warmup_steps, total_steps)]
for i in range(1, len(decay_lrs)):
    assert decay_lrs[i] <= decay_lrs[i - 1] + 1e-12, "LR should decrease during cosine decay"

# All LR values are non-negative
all_lrs = [warmup_cosine_lr(s, warmup_steps, total_steps, base_lr) for s in range(total_steps)]
assert all(lr >= 0 for lr in all_lrs), "All LR values should be non-negative"

# Returns float
assert isinstance(warmup_cosine_lr(0, 10, 100, 0.01), float)

# Different base_lr scales correctly
assert abs(warmup_cosine_lr(50, 100, 1000, 0.1) - 0.05) < 1e-10

print("Problem 8: ALL TESTS PASSED")

---

## Scoring Yourself

| Result | Assessment |
|--------|------------|
| 8/8 in < 20 min each | Ready for MEDIUM tier |
| 6-7/8 | Close — review the ones you missed, redo tomorrow |
| 4-5/8 | Gaps in fundamentals — drill these patterns daily |
| < 4/8 | Spend a week on PyTorch basics before moving on |